# AI Anomaly Detector - Credit Card Fraud Detection

## 1. Project Introduction
This internship notebook develops an unsupervised AI Anomaly Detector for the Credit Card Fraud Detection dataset (`creditcard.csv`). An anomaly is an observation that differs from the learned or expected pattern. Detecting unusual transactions is useful because suspicious activity can be reviewed before it is confirmed as fraud.

The dataset contains transaction time, amount, anonymized PCA-style features V1 through V28, and Class, where 0 means normal and 1 means fraud. Class is ground-truth information for evaluation only. Unlike supervised fraud classification, anomaly detection learns patterns without using fraud labels as model inputs. Isolation Forest is selected because it isolates unusual observations efficiently and is designed for unsupervised anomaly detection.


## 2. Import Libraries
Import the libraries required for data loading, exploration, preprocessing, Isolation Forest modeling, evaluation, visualization, and artifact export.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay,
)

print("Libraries imported successfully.")

## 3. Load Dataset
Load the actual creditcard.csv file with a relative path and inspect the first and last records.


In [ ]:
df = pd.read_csv("creditcard.csv")
print("First 5 rows:")
display(df.head())
print("Last 5 rows:")
display(df.tail())
print("Dataset shape:", df.shape)
print("Column names:", df.columns.tolist())

## 4. Dataset Information
Review the structure, dimensions, data types, numerical columns, categorical columns, and target column.


In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("
Dataset information:")
df.info()
print("
Statistical summary:")
display(df.describe().T)
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()
target_column = "Class"
print("Numerical columns:", len(numerical_columns))
print("Categorical columns:", categorical_columns)
print("Ground-truth column:", target_column)

## 5. Data Quality Report
Check missing values, duplicate rows, data types, unique values, and basic statistics. Duplicates are reported and retained unless they are invalid records; no data is silently removed.


In [ ]:
quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Missing_Values": df.isna().sum().values,
    "Unique_Values": df.nunique().values,
})
display(quality_report)
print("Total duplicate rows:", int(df.duplicated().sum()))
print("Total missing values:", int(df.isna().sum().sum()))
display(df.describe(include="all").T.head())

## 6. Class Distribution
Analyze the ground-truth Class column. Class 0 is Normal and Class 1 is Fraud; the imbalance is important because fraud is much rarer.


In [ ]:
class_counts = df["Class"].value_counts().sort_index()
class_percentages = df["Class"].value_counts(normalize=True).sort_index().mul(100)
class_distribution = pd.DataFrame({"Count": class_counts, "Percentage": class_percentages})
class_distribution.index = ["Normal", "Fraud"]
display(class_distribution)
fig = px.bar(class_distribution.reset_index(), x="index", y="Count", color="index", title="Normal vs Fraud Transactions", labels={"index":"Class"})
fig.show()

## 7. Exploratory Data Analysis
Explore transaction amount, time, normal-versus-fraud amounts, correlations, and representative feature distributions.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes[0, 0].hist(df["Amount"], bins=60, color="steelblue")
axes[0, 0].set_title("Transaction Amount Distribution")
axes[0, 0].set_xlabel("Amount")
axes[0, 0].set_ylabel("Frequency")
axes[0, 1].hist(df["Time"], bins=60, color="darkorange")
axes[0, 1].set_title("Transaction Time Distribution")
axes[0, 1].set_xlabel("Time")
axes[0, 1].set_ylabel("Frequency")
axes[1, 0].hist(df.loc[df["Class"] == 0, "Amount"], bins=50, alpha=0.7, label="Normal")
axes[1, 0].hist(df.loc[df["Class"] == 1, "Amount"], bins=50, alpha=0.7, label="Fraud")
axes[1, 0].set_title("Normal vs Fraud Amounts")
axes[1, 0].set_xlabel("Amount")
axes[1, 0].legend()
axes[1, 1].boxplot([df.loc[df["Class"] == 0, "Amount"], df.loc[df["Class"] == 1, "Amount"]], labels=["Normal", "Fraud"], showfliers=False)
axes[1, 1].set_title("Amount Comparison Without Extreme Fliers")
axes[1, 1].set_ylabel("Amount")
plt.tight_layout()
plt.show()

plt.figure(figsize=(16, 12))
plt.imshow(df.select_dtypes(include=np.number).corr(), cmap="coolwarm", aspect="auto", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.title("Numerical Feature Correlation Heatmap")
plt.xlabel("Feature Index")
plt.ylabel("Feature Index")
plt.show()

for feature in ["V1", "V2", "V3", "V4"]:
    fig = px.histogram(df, x=feature, color="Class", barmode="overlay", title=f"{feature} Distribution by Class")
    fig.show()

## 8. Prepare Data for Anomaly Detection
Separate X from y. Class is ground truth for evaluation only and is excluded from every Isolation Forest input.


In [ ]:
X = df.drop(columns=["Class"]).copy()
y = df["Class"].copy()
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class included in X?", "Class" in X.columns)

## 9. Preprocessing
Fill missing numerical values with training-safe medians and scale features with StandardScaler. The scaler is fit only on the training split to avoid leakage.


In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
feature_medians = X_train_raw.median()
X_train_clean = X_train_raw.fillna(feature_medians)
X_test_clean = X_test_raw.fillna(feature_medians)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test_clean)
print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)

## 10. Train/Test Strategy
Use a reproducible stratified split for evaluation. The model receives only scaled transaction features; y is used after prediction to measure performance.


In [ ]:
print("Training normal/fraud counts:")
print(y_train.value_counts().rename(index={0: "Normal", 1: "Fraud"}))
print("Testing normal/fraud counts:")
print(y_test.value_counts().rename(index={0: "Normal", 1: "Fraud"}))

## 11. Isolation Forest Model
Isolation Forest isolates observations through random feature splits. Unusual points need fewer splits, so they receive stronger anomaly scores. Contamination represents the expected proportion of anomalies and remains configurable.


In [ ]:
CONTAMINATION = 0.05
model = IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=42, n_jobs=-1)
model.fit(X_train_scaled)
print(f"Isolation Forest trained with contamination={CONTAMINATION}.")

## 12. Generate Predictions
Isolation Forest returns 1 for normal and -1 for anomaly. Convert those values into readable prediction fields and preserve Class only as ground truth.


In [ ]:
test_raw_predictions = model.predict(X_test_scaled)
test_predictions = np.where(test_raw_predictions == -1, 1, 0)
test_status = np.where(test_predictions == 1, "Potential Anomaly", "Normal")
test_results = X_test_raw.copy()
test_results["Class"] = y_test
test_results["Anomaly_Prediction"] = test_predictions
test_results["Anomaly_Status"] = test_status
print(pd.Series(test_status).value_counts())

## 13. Generate Anomaly Scores
Use the negative decision function so larger Anomaly_Score values indicate records that are more suspicious according to the fitted model.


In [ ]:
test_results["Anomaly_Score"] = -model.decision_function(X_test_scaled)
print(test_results["Anomaly_Score"].describe())
fig = px.histogram(test_results, x="Anomaly_Score", color="Anomaly_Status", title="Anomaly Score Distribution")
fig.show()

## 14. Top Suspicious Records
Rank records by anomaly score and display the most suspicious observations. A detected anomaly is a Potential Anomaly, not automatically confirmed fraud.


In [ ]:
top_suspicious = test_results.sort_values("Anomaly_Score", ascending=False).head(20)
display(top_suspicious[["Time", "Amount", "Class", "Anomaly_Score", "Anomaly_Status"]])

## 15. Anomaly Statistics
Summarize normal observations and Potential Anomalies detected in the test set.


In [ ]:
anomaly_statistics = pd.DataFrame({"Metric": ["Total Records", "Normal Records", "Detected Anomalies", "Anomaly Percentage"], "Value": [len(test_results), int((test_predictions == 0).sum()), int((test_predictions == 1).sum()), float(test_predictions.mean() * 100)]})
display(anomaly_statistics)
count_chart = pd.DataFrame({"Status": ["Normal", "Potential Anomaly"], "Count": [(test_predictions == 0).sum(), (test_predictions == 1).sum()]})
px.bar(count_chart, x="Status", y="Count", color="Status", title="Normal vs Potential Anomaly Count").show()

## 16. Model Evaluation
Evaluate with precision, recall, F1-score, ROC-AUC, PR-AUC, a classification report, and a confusion matrix. Accuracy is not emphasized because the classes are highly imbalanced.


In [ ]:
y_true = y_test.to_numpy()
precision = precision_score(y_true, test_predictions, zero_division=0)
recall = recall_score(y_true, test_predictions, zero_division=0)
f1 = f1_score(y_true, test_predictions, zero_division=0)
roc_auc = roc_auc_score(y_true, test_results["Anomaly_Score"])
pr_auc = average_precision_score(y_true, test_results["Anomaly_Score"])
cm = confusion_matrix(y_true, test_predictions)
evaluation_metrics = pd.DataFrame({"Metric": ["Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC"], "Value": [precision, recall, f1, roc_auc, pr_auc]})
display(evaluation_metrics)
print(classification_report(y_true, test_predictions, target_names=["Normal", "Fraud"], zero_division=0))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Fraud"]).plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

## 17. Contamination / Threshold Analysis
Compare several contamination values. Higher contamination generally marks more records as anomalies, changing the precision-recall tradeoff. Select the value with the strongest F1-score, using PR-AUC as a tie-breaker.


In [ ]:
comparison_rows = []
for contamination_value in [0.01, 0.02, 0.05, 0.10]:
    candidate = IsolationForest(n_estimators=200, contamination=contamination_value, random_state=42, n_jobs=-1)
    candidate.fit(X_train_scaled)
    candidate_raw = candidate.predict(X_test_scaled)
    candidate_pred = np.where(candidate_raw == -1, 1, 0)
    candidate_scores = -candidate.decision_function(X_test_scaled)
    comparison_rows.append({"Contamination": contamination_value, "Detected_Anomalies": int(candidate_pred.sum()), "Precision": precision_score(y_true, candidate_pred, zero_division=0), "Recall": recall_score(y_true, candidate_pred, zero_division=0), "F1": f1_score(y_true, candidate_pred, zero_division=0), "ROC_AUC": roc_auc_score(y_true, candidate_scores), "PR_AUC": average_precision_score(y_true, candidate_scores)})
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)
comparison_df.set_index("Contamination")[["Precision", "Recall", "F1", "PR_AUC"]].plot(marker="o", figsize=(10, 5), title="Contamination Comparison")
plt.ylabel("Metric Value")
plt.show()
selected_contamination = float(comparison_df.sort_values(["F1", "PR_AUC"], ascending=False).iloc[0]["Contamination"])
print("Selected contamination:", selected_contamination)

## 18. 2D Feature Visualization
Visualize two numerical features and color points by Normal or Potential Anomaly. This view is only a projection and does not represent all dataset dimensions.


In [ ]:
visual_df = test_results.copy()
visual_df["Visualization_Status"] = visual_df["Anomaly_Status"]
fig = px.scatter(visual_df.sample(n=min(10000, len(visual_df)), random_state=42), x="V1", y="V2", color="Visualization_Status", hover_data=["Amount", "Anomaly_Score", "Class"], title="2D V1 vs V2 Potential Anomalies", opacity=0.65)
fig.show()

## 19. Final Model
Train the final Isolation Forest on all preprocessed records using the selected contamination value. Keep the scaler available for the future Streamlit application.


In [ ]:
X_clean = X.fillna(X.median())
X_scaled_full = scaler.fit_transform(X_clean)
final_model = IsolationForest(n_estimators=200, contamination=selected_contamination, random_state=42, n_jobs=-1)
final_model.fit(X_scaled_full)
full_raw_predictions = final_model.predict(X_scaled_full)
full_scores = -final_model.decision_function(X_scaled_full)
final_results = df.copy()
final_results["Anomaly_Score"] = full_scores
final_results["Anomaly_Prediction"] = np.where(full_raw_predictions == -1, 1, 0)
final_results["Anomaly_Status"] = np.where(full_raw_predictions == -1, "Potential Anomaly", "Normal")
print("Final model trained on all records.")

## 20. Save Model and Preprocessing
Save the final Isolation Forest and scaler with Joblib for reuse by a separate application.


In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(final_model, "models/isolation_forest.pkl")
joblib.dump(scaler, "models/scaler.pkl")
print("Saved models/isolation_forest.pkl and models/scaler.pkl")

## 21. Export Results
Export complete results and only the detected Potential Anomalies to the outputs folder.


In [ ]:
os.makedirs("outputs", exist_ok=True)
final_results.to_csv("outputs/anomaly_detection_results.csv", index=False)
detected_anomalies = final_results[final_results["Anomaly_Prediction"] == 1].copy()
detected_anomalies.to_csv("outputs/detected_anomalies.csv", index=False)
print("Saved outputs/anomaly_detection_results.csv")
print("Saved outputs/detected_anomalies.csv")

## 22. Final Summary
This notebook used the actual Credit Card Fraud Detection dataset, checked its quality, explored transaction behavior, excluded Class from model inputs, filled missing values with training medians, scaled features, and trained Isolation Forest. The selected contamination value was chosen dynamically from the comparison results. Evaluation used precision, recall, F1-score, ROC-AUC, PR-AUC, and the confusion matrix.

Limitations include severe class imbalance, anonymized feature meanings, and the fact that model thresholds may not transfer unchanged to a different transaction population. An anomaly is an observation that differs from the learned or expected pattern; it should not automatically be considered confirmed fraud.
